In [1]:
!pip install langgraph

In [2]:
import os
from dotenv import load_dotenv
from backend.langgraph_agents import run_chatbot

load_dotenv()

print("Supabase URL:", os.environ.get("SUPABASE_URL"))
print("OpenAI key loaded?:", bool(os.environ.get("OPENAI_API_KEY")))

Supabase URL: https://zsiaodpoyplzfqommhyf.supabase.co
OpenAI key loaded?: True


In [3]:
query = "What are the symptoms of social anxiety disorder?"
result = run_chatbot(query, k=5)

# Access docs from the dict
docs = result.get("docs", [])  # <-- safe access

print("\nRetrieved chunks:\n")
for i, d in enumerate(docs, 1):
    print(f"{i}. {d['content'][:300]}... (source: {d['source']})")



Retrieved chunks:

1. end up avoiding places or events that cause distress or generate feelings of embarrassment. In some cases, anxiety may arise only during performance situations such as giving a speech, competing in a sports game, or playing a musical instrument on stage. Social anxiety disorder usually starts during... (source: Social Anxiety Disorder_ What You Need to Know - National Institute of Mental Health (NIMH).pdf)
2. stomach Have a rigid body posture or speak with an overly soft voice Find it difficult to make eye contact, be around people they don’t know, or talk to people in social situations, even when they want to Feel self-consciousness or fear that people will judge them negatively Avoid places where there... (source: Social Anxiety Disorder_ What You Need to Know - National Institute of Mental Health (NIMH).pdf)
3. The National Institute of Mental Health: https://www.nimh.nih.gov/health/publications/social-anxiety-disorder-more-than-just-shyness Social Anxiety Dis

In [4]:
# --- 2. GPT answer
context_text = "\n\n".join([d["content"] for d in docs])
from backend.utils.supabase_client import client as openai_client

completion = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant with access to a knowledge base."},
        {"role": "user", "content": f"Answer the question based on context below.\n\nContext:\n{context_text}\n\nQuestion: {query}"}
    ]
)

print("\nGPT Answer:\n")
print(completion.choices[0].message.content)



GPT Answer:

The symptoms of social anxiety disorder include:

- Blushing, sweating, or trembling.
- Rapid heart rate.
- Feeling their “mind going blank” or feeling sick to their stomach.
- Rigid body posture or speaking with an overly soft voice.
- Difficulty making eye contact or being around people they don’t know.
- Difficulty talking to people in social situations, even when they want to.
- Feelings of self-consciousness or fear of negative judgment.
- Avoidance of places where there are other people or situations requiring interaction.
- Analyzing their performance and identifying self-perceived flaws.
- Expecting the worst possible consequences from negative social situations.

For children and teens, additional symptoms may include:

- Avoiding school.
- Difficulty making friends.
- Complaining of stomachaches or headaches.
- Emotional outbursts in social situations.
- Refusing to speak or participate in social situations.

Overall, symptoms may fluctuate over time and can wor

In [7]:
import pandas as pd
import json

os.makedirs("outputs", exist_ok=True)

# Select only relevant keys for CSV
csv_docs = [
    {
        "id": d.get("id"),
        "source": d.get("source"),
        "content": d.get("content"),
        "distance": d.get("distance")
    }
    for d in docs
]

# Save retrieved docs to CSV
df = pd.DataFrame(csv_docs)
df.to_csv("outputs/langgraph_test_results.csv", index=False)

# Save log (full JSON including query)
with open("outputs/langgraph_test_log.json", "w") as f:
    json.dump({"query": query, "retrieved": docs}, f, indent=2)

print("\nSaved outputs to outputs/")


Saved outputs to outputs/
